[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 Easy: Kaiming Initialization

Implement **Kaiming (He) normal initialization** for weight tensors.

$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}$$

### Signature
```python
def kaiming_init(weight: Tensor) -> Tensor:
    # Initialize weight in-place with Kaiming normal
    # fan_in = weight.shape[1]
    # Returns the weight tensor
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00


In [2]:
import torch
import math

In [8]:
'''
🎯 Kaiming 的作用

👉 保持 variance ≈ constant across layers


weight = (输出维度， 输入维度)
eg. weight = (64, 128) it means 每一层都要看128个输入

直觉： 如果有一层线性层： 根据输入维度 fan_in 来控制权重方差，让每层输出的方差保持在合理范围
因为 ReLU 会把一半左右的值截断成 0，所以它比普通的 Xavier 初始化多了一个 2

Kaiming 初始化主要给 ReLU 网络 用
'''

def kaiming_init(weight):
  fan_in = weight.shape[1] # 每个输出神经元接收多少个输入。
  std = math.sqrt(2 / fan_in)

# 因为初始化权重只是“赋初值”，不是训练过程中的可微计算，所以不需要梯度追踪。
  with torch.no_grad():
    weight.normal_(mean=0.0, std=std) # 这是 PyTorch 的 原地操作。 注意下划线_
    # normal_()：原地修改
    # normal()：通常返回新 tensor
    # 这里它会把 weight 里面每个元素直接替换成一个从正态分布采样出来的随机数

    # 虽然 weight 已经被原地改掉了，但还是返回一下
  return weight

In [9]:
# 🧪 Debug
import math
w = torch.empty(256, 512)
kaiming_init(w)
print(f'Mean: {w.mean():.4f} (expect ~0)')
print(f'Std:  {w.std():.4f} (expect {math.sqrt(2/512):.4f})')

Mean: -0.0002 (expect ~0)
Std:  0.0627 (expect 0.0625)


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('weight_init')


🧪 Testing: Kaiming Initialization (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Mean approximately 0 (4.1ms)
  ✅ [2/4] Std matches sqrt(2/fan_in) (3.3ms)
  ✅ [3/4] Returns same tensor (in-place) (0.2ms)
  ✅ [4/4] Smaller fan_in gives larger std (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.0ms total)
  Progress saved. Run status() to see your dashboard.

